# Data Pull from Alpha Vantage
**AAI-590 Capstone | Validex Growth Investors**


### Purpose
This notebook pulls **all raw data** from the Alpha Vantage API and saves it locally.


### What gets pulled
| Section | Endpoint | Output |
|---|---|---|
| 1 | **TIME_SERIES_DAILY_ADJUSTED** | Daily OHLCV + dividends + splits |
| 2 | **INCOME_STATEMENT**| Quarterly income statements |
| 2 | **BALANCE_SHEET** | Quarterly balance sheets |
| 2 | **CASH_FLOW** | Quarterly cash flow statements |
| 2 | **OVERVIEW** | Company metadata & static ratios |
| 3 | **HISTORICAL_OPTIONS** | Options chain (premium key recommended) |

### Output structure
```
data/
├── raw/
│   ├── daily_adjusted/          --> raw JSON per ticker
│   ├── fundamentals/
│   │   ├── income_statement/
│   │   ├── balance_sheet/
│   │   ├── cash_flow/
│   │   └── overview/
│   └── options/                 --> raw JSON per ticker
└── processed/
    ├── daily_adjusted/          --> parsed parquet + csv per ticker + combined
    ├── fundamentals/            --> parsed parquet + csv per statement type
    └── options/                 --> parsed parquet + csv per ticker + combined
```

## Section 0 — Configuration & Setup

In [1]:
import os
import time
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from dotenv import load_dotenv

print(" Imports OK")

 Imports OK


In [2]:
# API KEY
#   ALPHAVANTAGE_API_KEY=your_key_here
load_dotenv()
API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
if not API_KEY:
    raise RuntimeError(" Missing ALPHAVANTAGE_API_KEY. Add it to your .env file.")
print(f"API key loaded: {API_KEY[:4]}...{API_KEY[-4:]}")

API key loaded: QWPX...6I4L


In [ ]:
# Universe & Date Range 
TICKERS    = ["ADMA", "NTRA", "AXON", "SHAK", "AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META"]
DATE_START = "2005-01-01"
DATE_END   = "2025-12-31"
BASE_URL   = "https://www.alphavantage.co/query"

#  Directory Structure 
DIRS = {
    "raw_daily"        : Path("data/raw/daily_adjusted"),
    "raw_income"       : Path("data/raw/fundamentals/income_statement"),
    "raw_balance"      : Path("data/raw/fundamentals/balance_sheet"),
    "raw_cashflow"     : Path("data/raw/fundamentals/cash_flow"),
    "raw_overview"     : Path("data/raw/fundamentals/overview"),
    "raw_options"      : Path("data/raw/options"),
    "proc_daily"       : Path("data/processed/daily_adjusted"),
    "proc_fundamentals": Path("data/processed/fundamentals"),
    "proc_options"     : Path("data/processed/options"),
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"Universe  : {TICKERS}")
print(f"Date range: {DATE_START} → {DATE_END}")
print(f"Directories created")

Universe  : ['ADMA', 'NTRA', 'AXON', 'SHAK', 'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOG', 'META']
Date range: 1985-01-01 → 2025-12-31
Directories created


In [4]:
#  Generic API Helper with File-Based Caching 
def av_get(params: dict, cache_path: Path, sleep_s: float = 15.0) -> dict:
    """
    Fetches data from Alpha Vantage with local JSON caching.
    - If cache_path exists: loads from disk (no API call made).
    - If not: calls the API, validates response, saves JSON, then sleeps.

    sleep_s: pause after each live call (free tier = 25 calls/day, premium = 75/min).
    """
    if cache_path.exists():
        print(f"    [cache] {cache_path.name}")
        return json.loads(cache_path.read_text())

    full_params = {**params, "apikey": API_KEY}
    r = requests.get(BASE_URL, params=full_params, timeout=60)
    r.raise_for_status()
    data = r.json()

    # Alpha Vantage error handling
    if "Note"          in data: raise RuntimeError(f"Rate limit hit: {data['Note']}")
    if "Information"   in data: raise RuntimeError(f"Quota message: {data['Information']}")
    if "Error Message" in data: raise RuntimeError(f"API error: {data['Error Message']}")

    cache_path.write_text(json.dumps(data))
    print(f"    [api]   {cache_path.name}  → sleeping {sleep_s}s")
    time.sleep(sleep_s)
    return data

print(" av_get helper defined")

 av_get helper defined



## Section 1 — Daily Adjusted Price Data
TIME_SERIES_DAILY_ADJUSTED --> open, high, low, close, adjusted close, volume, dividend, split coefficient

In [5]:
def parse_daily_adjusted(symbol: str, data: dict,
                         date_start: str = DATE_START,
                         date_end:   str = DATE_END) -> pd.DataFrame:
    ts_key = "Time Series (Daily)"
    if ts_key not in data:
        raise KeyError(f"{symbol}: '{ts_key}' missing. Keys found: {list(data.keys())}")

    df = pd.DataFrame.from_dict(data[ts_key], orient="index")
    df = df.rename(columns={
        "1. open"             : "open",
        "2. high"             : "high",
        "3. low"              : "low",
        "4. close"            : "close",
        "5. adjusted close"   : "adj_close",
        "6. volume"           : "volume",
        "7. dividend amount"  : "dividend",
        "8. split coefficient": "split_coeff",
    })

    df.index = pd.to_datetime(df.index)
    df = df.sort_index().loc[date_start:date_end]   # filter to project range

    for c in ["open","high","low","close","adj_close","volume","dividend","split_coeff"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df.insert(0, "symbol", symbol)
    df.reset_index(names="date", inplace=True)
    return df

print(" parse_daily_adjusted defined")

 parse_daily_adjusted defined


In [6]:
# ─ Pull Daily Adjusted for All Tickers 
daily_dfs    = []
daily_errors = {}

for sym in tqdm(TICKERS, desc="Daily Adjusted"):
    try:
        cache = DIRS["raw_daily"] / f"{sym}_daily_adjusted_full.json"
        data  = av_get(
            {"function": "TIME_SERIES_DAILY_ADJUSTED", "symbol": sym, "outputsize": "full"},
            cache_path=cache,
            sleep_s=15.0,
        )
        df = parse_daily_adjusted(sym, data)

        # Save per ticker
        df.to_parquet(DIRS["proc_daily"] / f"{sym}_daily_adjusted.parquet", index=False)
        df.to_csv(DIRS["proc_daily"]    / f"{sym}_daily_adjusted.csv",     index=False)

        daily_dfs.append(df)
        print(f"   {sym}: {len(df):,} rows  [{df['date'].min().date()} → {df['date'].max().date()}]")

    except Exception as e:
        daily_errors[sym] = str(e)
        print(f"   {sym}: {e}")

print(f"\nDaily → Success: {len(daily_dfs)} | Failures: {len(daily_errors)}")
if daily_errors:
    print(daily_errors)

Daily Adjusted:   0%|          | 0/10 [00:00<?, ?it/s]

    [api]   ADMA_daily_adjusted_full.json  → sleeping 15.0s
   ADMA: 3,070 rows  [2013-10-17 → 2025-12-31]
    [api]   NTRA_daily_adjusted_full.json  → sleeping 15.0s
   NTRA: 2,641 rows  [2015-07-02 → 2025-12-31]
    [api]   AXON_daily_adjusted_full.json  → sleeping 15.0s
   AXON: 6,179 rows  [2001-06-07 → 2025-12-31]
    [api]   SHAK_daily_adjusted_full.json  → sleeping 15.0s
   SHAK: 2,747 rows  [2015-01-30 → 2025-12-31]
    [api]   AAPL_daily_adjusted_full.json  → sleeping 15.0s
   AAPL: 6,582 rows  [1999-11-01 → 2025-12-31]
    [api]   MSFT_daily_adjusted_full.json  → sleeping 15.0s
   MSFT: 6,582 rows  [1999-11-01 → 2025-12-31]
    [api]   NVDA_daily_adjusted_full.json  → sleeping 15.0s
   NVDA: 6,582 rows  [1999-11-01 → 2025-12-31]
    [api]   AMZN_daily_adjusted_full.json  → sleeping 15.0s
   AMZN: 6,582 rows  [1999-11-01 → 2025-12-31]
    [api]   GOOG_daily_adjusted_full.json  → sleeping 15.0s
   GOOG: 2,960 rows  [2014-03-27 → 2025-12-31]
    [api]   META_daily_adjusted_full.

In [7]:
#  Combine & Save All Tickers 

combined_daily = pd.concat(daily_dfs, ignore_index=True)
combined_daily["date"] = pd.to_datetime(combined_daily["date"])

combined_daily.to_parquet(DIRS["proc_daily"] / "ALL_daily_adjusted.parquet", index=False)
combined_daily.to_csv(DIRS["proc_daily"]    / "ALL_daily_adjusted.csv",     index=False)

print(f"Combined daily shape: {combined_daily.shape}")
combined_daily.groupby("symbol")["date"].agg(["min", "max", "count"])

Combined daily shape: (47350, 10)


,min,max,count
symbol,,,
AAPL,1999-11-01,2025-12-31,6582
ADMA,2013-10-17,2025-12-31,3070
AMZN,1999-11-01,2025-12-31,6582
AXON,2001-06-07,2025-12-31,6179
GOOG,2014-03-27,2025-12-31,2960
META,2012-05-18,2025-12-31,3425
MSFT,1999-11-01,2025-12-31,6582
NTRA,2015-07-02,2025-12-31,2641
NVDA,1999-11-01,2025-12-31,6582



## Section 2 — Fundamentals Data
Pulls quarterly **Income Statement**, **Balance Sheet**, **Cash Flow**, and **Company Overview**.

In [8]:
def pull_fundamental(symbol: str, function: str, raw_dir: Path) -> pd.DataFrame:
    """
    Pulls INCOME_STATEMENT, BALANCE_SHEET, or CASH_FLOW quarterly reports.
    Returns a DataFrame — one row per quarter.
    """
    cache = raw_dir / f"{symbol}_{function.lower()}.json"
    data  = av_get({"function": function, "symbol": symbol},
                   cache_path=cache, sleep_s=15.0)

    if "quarterlyReports" not in data:
        raise KeyError(f"{symbol}/{function}: 'quarterlyReports' missing. Keys: {list(data.keys())}")

    df = pd.DataFrame(data["quarterlyReports"])
    df.insert(0, "symbol", symbol)
    df["fiscalDateEnding"] = pd.to_datetime(df["fiscalDateEnding"])

    # Convert all value columns to numeric
    for col in df.columns:
        if col not in ["symbol", "fiscalDateEnding", "reportedCurrency"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values("fiscalDateEnding").reset_index(drop=True)


def pull_overview(symbol: str) -> pd.Series:
    """
    Pulls company overview: sector, market cap, P/E, P/S, beta, 52-week range, etc.
    Returns a Series (one row per company).
    """
    cache = DIRS["raw_overview"] / f"{symbol}_overview.json"
    data  = av_get({"function": "OVERVIEW", "symbol": symbol},
                   cache_path=cache, sleep_s=15.0)
    return pd.Series(data)

print(" Fundamental pull helpers defined")

 Fundamental pull helpers defined


In [9]:
# Pull All Fundamental Statements 
income_dfs    = []
balance_dfs   = []
cashflow_dfs  = []
overview_list = []
fund_errors   = {}

FUND_MAP = {
    "INCOME_STATEMENT": (DIRS["raw_income"],   income_dfs),
    "BALANCE_SHEET"   : (DIRS["raw_balance"],  balance_dfs),
    "CASH_FLOW"       : (DIRS["raw_cashflow"], cashflow_dfs),
}

for sym in tqdm(TICKERS, desc="Fundamentals"):
    try:
        for func, (raw_dir, df_list) in FUND_MAP.items():
            df = pull_fundamental(sym, func, raw_dir)
            df_list.append(df)

        overview_list.append(pull_overview(sym))
        print(f" {sym}: income / balance / cashflow / overview pulled")

    except Exception as e:
        fund_errors[sym] = str(e)
        print(f"  {sym}: {e}")

print(f"\nFundamentals → Success: {len(income_dfs)} | Failures: {len(fund_errors)}")
if fund_errors:
    print(fund_errors)

Fundamentals:   0%|          | 0/10 [00:00<?, ?it/s]

    [api]   ADMA_income_statement.json  → sleeping 15.0s
    [api]   ADMA_balance_sheet.json  → sleeping 15.0s
    [api]   ADMA_cash_flow.json  → sleeping 15.0s
    [api]   ADMA_overview.json  → sleeping 15.0s
 ADMA: income / balance / cashflow / overview pulled
    [api]   NTRA_income_statement.json  → sleeping 15.0s
    [api]   NTRA_balance_sheet.json  → sleeping 15.0s
    [api]   NTRA_cash_flow.json  → sleeping 15.0s


KeyboardInterrupt: 

In [10]:
# Combine & Save Fundamentals 
income_all   = pd.concat(income_dfs,   ignore_index=True)
balance_all  = pd.concat(balance_dfs,  ignore_index=True)
cashflow_all = pd.concat(cashflow_dfs, ignore_index=True)
overview_all = pd.DataFrame(overview_list) if overview_list else pd.DataFrame()

for name, df in [
    ("income_statement", income_all),
    ("balance_sheet",    balance_all),
    ("cash_flow",        cashflow_all),
]:
    df.to_parquet(DIRS["proc_fundamentals"] / f"ALL_{name}.parquet", index=False)
    df.to_csv(DIRS["proc_fundamentals"]    / f"ALL_{name}.csv",     index=False)
    print(f"  Saved {name}: {df.shape}")

if not overview_all.empty:
    overview_all.to_csv(DIRS["proc_fundamentals"] / "ALL_overview.csv", index=False)
    print(f"  Saved overview: {overview_all.shape}")

print("\n All fundamentals saved")

  Saved income_statement: (736, 27)
  Saved balance_sheet: (729, 39)
  Saved cash_flow: (731, 30)
  Saved overview: (10, 55)

 All fundamentals saved


## Section 3 — Historical Options Data

### How the endpoint works
Alpha Vantage **HISTORICAL_OPTIONS** returns **one full options chain per call** (all strikes + expirations for a given ticker on a given date).  
There is no bulk/range parameter — each call = one ticker + one date.

### Strategy used here
Since the project runs at **monthly decision frequency**, we pull one snapshot per month per ticker (first trading day of each month).  
- **10 tickers X ~132 months (2015–2025) = ~1,320 API calls**
- At 75 calls/min (premium), this takes **~18 minutes** on the first run
- All responses are **cached as JSON** — subsequent runs are instant

### Cache structure

```
data/raw/options/
└── {SYMBOL}/
    ├── {SYMBOL}_2015-01-02.json
    ├── {SYMBOL}_2015-02-02.json
    └── ...
```

In [11]:
# Generate Monthly Decision Dates 
# First trading day of each month between DATE_START and DATE_END.
# We use pd.bdate_range to get business days, then resample to month-start.

all_bdays = pd.bdate_range(start=DATE_START, end=DATE_END, freq="B")
monthly_dates = (
    pd.Series(all_bdays)
    .groupby(pd.Series(all_bdays).dt.to_period("M"))
    .first()
    .values
)
monthly_dates = pd.DatetimeIndex(monthly_dates)

print(f"   Monthly decision dates: {len(monthly_dates)} dates")
print(f"   First: {monthly_dates[0].date()}  |  Last: {monthly_dates[-1].date()}")
print(f"   Sample: {[str(d.date()) for d in monthly_dates[:6]]} ...")

   Monthly decision dates: 132 dates
   First: 2015-01-01  |  Last: 2025-12-01
   Sample: ['2015-01-01', '2015-02-02', '2015-03-02', '2015-04-01', '2015-05-01', '2015-06-01'] ...


In [18]:
#  Options Parse Helper 
def parse_options_response(symbol: str, date_str: str, data: dict) -> pd.DataFrame:
    """
    Parses a single HISTORICAL_OPTIONS API response for one (symbol, date).
    Column names matched to actual AV response format.
    """
    if "data" not in data or not data["data"]:
        return pd.DataFrame()

    df = pd.DataFrame(data["data"])
    df.insert(0, "symbol", symbol)

    # ── Rename to standardized internal names ─────────────────────────────────
    # Based on actual AV response: contractID, symbol, expiration, strike,
    # type, last, mark, bid, bid_size, ask, ask_size, volume, open_interest,
    # date, implied_volatility, delta, gamma, theta, vega, rho
    rename_map = {
        "date"              : "trade_date",      # actual field name from AV
        "type"              : "call_put",
        "implied_volatility": "implied_vol",     # AV uses implied_volatility not impliedVolatility
        "open_interest"     : "open_interest",   # already correct, just explicit
        "bid_size"          : "bid_size",
        "ask_size"          : "ask_size",
        "mark"              : "mark",
    }
    for src, tgt in rename_map.items():
        if src in df.columns and tgt not in df.columns:
            df = df.rename(columns={src: tgt})

    # Ensure trade_date exists (fallback to requested date_str)
    if "trade_date" not in df.columns:
        df["trade_date"] = date_str

    # ── Parse dates ───────────────────────────────────────────────────────────
    for col in ["trade_date", "expiration"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # ── Parse numerics ────────────────────────────────────────────────────────
    num_cols = ["strike", "last", "mark", "bid", "ask",
                "bid_size", "ask_size", "volume", "open_interest",
                "implied_vol", "delta", "gamma", "theta", "vega", "rho"]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.reset_index(drop=True)

In [27]:
# Check which tickers have options data available
import requests, json

check_date = "2024-01-02"
results = {}

for sym in TICKERS:
    r = requests.get(
        "https://www.alphavantage.co/query",
        params={
            "function" : "HISTORICAL_OPTIONS",
            "symbol"   : sym,
            "date"     : check_date,
            "apikey"   : API_KEY
        },
        timeout=30
    )
    data = r.json()
    row_count = len(data.get("data", []))
    results[sym] = row_count
    print(f"  {sym:6s} : {row_count:,} option rows")
    time.sleep(1.0)  # respect rate limit

print("\nSummary:")
print(f"  Tickers WITH options data  : {[s for s,v in results.items() if v > 0]}")
print(f"  Tickers WITHOUT options data: {[s for s,v in results.items() if v == 0]}")

  ADMA   : 24 option rows
  NTRA   : 120 option rows
  AXON   : 538 option rows
  SHAK   : 814 option rows
  AAPL   : 1,896 option rows
  MSFT   : 2,788 option rows
  NVDA   : 5,140 option rows
  AMZN   : 1,766 option rows
  GOOG   : 1,556 option rows
  META   : 3,798 option rows

Summary:
  Tickers WITH options data  : ['ADMA', 'NTRA', 'AXON', 'SHAK', 'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOG', 'META']
  Tickers WITHOUT options data: []


In [ ]:
# import shutil

# # ── Clear bad option cache and re-run ─────────────────────────────────────────
# # Only run this once to wipe the previously cached (broken) responses
# for sym in TICKERS:
#     sym_dir = DIRS["raw_options"] / sym
#     if sym_dir.exists():
#         shutil.rmtree(sym_dir)
#         print(f"  Cleared cache: {sym_dir}")

# print(" Cache cleared — re-run the options pull loop now")

 Cache cleared — re-run the options pull loop now


In [31]:
# Pull Options — Monthly Loop Per Ticker 
#
# Each call --> one ticker + one date --> full options chain for that day
# Cache path: data/raw/options/{SYMBOL}/{SYMBOL}_{DATE}.json
#
# SLEEP_S: seconds between live API calls
#   Premium 75/min  --> 0.8s is safe (we use 1.0s for headroom)
#   Premium 150/min --> 0.5s is safe
#   Adjust based on the plan.
SLEEP_S = 1.0

opt_dfs    = []
opt_errors = {}   # {(symbol, date): error_message}

for sym in tqdm(TICKERS, desc="Tickers", position=0):

    sym_dir = DIRS["raw_options"] / sym
    sym_dir.mkdir(parents=True, exist_ok=True)

    sym_dfs = []

    for dt in tqdm(monthly_dates, desc=f"  {sym}", position=1, leave=False):
        date_str  = dt.strftime("%Y-%m-%d")
        cache     = sym_dir / f"{sym}_{date_str}.json"

        try:
            data = av_get(
                {"function": "HISTORICAL_OPTIONS", "symbol": sym, "date": date_str},
                cache_path=cache,
                sleep_s=SLEEP_S,
            )
            df = parse_options_response(sym, date_str, data)
            if not df.empty:
                sym_dfs.append(df)

        except Exception as e:
            opt_errors[(sym, date_str)] = str(e)

    if sym_dfs:
        sym_df = pd.concat(sym_dfs, ignore_index=True)
        sym_df.to_parquet(DIRS["proc_options"] / f"{sym}_options.parquet", index=False)
        sym_df.to_csv(DIRS["proc_options"]    / f"{sym}_options.csv",     index=False)
        opt_dfs.append(sym_df)
        print(f"   {sym}: {len(sym_df):,} rows across {len(sym_dfs)} months")
    else:
        print(f"    {sym}: no options data returned")

print(f"\nOptions pull complete.")
print(f"  Tickers with data : {len(opt_dfs)}")
print(f"  Failed calls      : {len(opt_errors)}")
if opt_errors:
    print("  Sample errors:", dict(list(opt_errors.items())[:5]))

Tickers:   0%|          | 0/10 [00:00<?, ?it/s]

  ADMA:   0%|          | 0/132 [00:00<?, ?it/s]

    [api]   ADMA_2015-01-01.json  → sleeping 1.0s
    [api]   ADMA_2015-02-02.json  → sleeping 1.0s
    [api]   ADMA_2015-03-02.json  → sleeping 1.0s
    [api]   ADMA_2015-04-01.json  → sleeping 1.0s
    [api]   ADMA_2015-05-01.json  → sleeping 1.0s
    [api]   ADMA_2015-06-01.json  → sleeping 1.0s
    [api]   ADMA_2015-07-01.json  → sleeping 1.0s
    [api]   ADMA_2015-08-03.json  → sleeping 1.0s
    [api]   ADMA_2015-09-01.json  → sleeping 1.0s
    [api]   ADMA_2015-10-01.json  → sleeping 1.0s
    [api]   ADMA_2015-11-02.json  → sleeping 1.0s
    [api]   ADMA_2015-12-01.json  → sleeping 1.0s
    [api]   ADMA_2016-01-01.json  → sleeping 1.0s
    [api]   ADMA_2016-02-01.json  → sleeping 1.0s
    [api]   ADMA_2016-03-01.json  → sleeping 1.0s
    [api]   ADMA_2016-04-01.json  → sleeping 1.0s
    [api]   ADMA_2016-05-02.json  → sleeping 1.0s
    [api]   ADMA_2016-06-01.json  → sleeping 1.0s
    [api]   ADMA_2016-07-01.json  → sleeping 1.0s
    [api]   ADMA_2016-08-01.json  → sleeping 1.0s


  NTRA:   0%|          | 0/132 [00:00<?, ?it/s]

    [api]   NTRA_2015-01-01.json  → sleeping 1.0s
    [api]   NTRA_2015-02-02.json  → sleeping 1.0s
    [api]   NTRA_2015-03-02.json  → sleeping 1.0s
    [api]   NTRA_2015-04-01.json  → sleeping 1.0s
    [api]   NTRA_2015-05-01.json  → sleeping 1.0s
    [api]   NTRA_2015-06-01.json  → sleeping 1.0s
    [api]   NTRA_2015-07-01.json  → sleeping 1.0s
    [api]   NTRA_2015-08-03.json  → sleeping 1.0s
    [api]   NTRA_2015-09-01.json  → sleeping 1.0s
    [api]   NTRA_2015-10-01.json  → sleeping 1.0s
    [api]   NTRA_2015-11-02.json  → sleeping 1.0s
    [api]   NTRA_2015-12-01.json  → sleeping 1.0s
    [api]   NTRA_2016-01-01.json  → sleeping 1.0s
    [api]   NTRA_2016-02-01.json  → sleeping 1.0s
    [api]   NTRA_2016-03-01.json  → sleeping 1.0s
    [api]   NTRA_2016-04-01.json  → sleeping 1.0s
    [api]   NTRA_2016-05-02.json  → sleeping 1.0s
    [api]   NTRA_2016-06-01.json  → sleeping 1.0s
    [api]   NTRA_2016-07-01.json  → sleeping 1.0s
    [api]   NTRA_2016-08-01.json  → sleeping 1.0s


...

    [api]   AMZN_2015-01-01.json  → sleeping 1.0s
    [api]   AMZN_2015-02-02.json  → sleeping 1.0s
    [api]   AMZN_2015-03-02.json  → sleeping 1.0s
    [api]   AMZN_2015-04-01.json  → sleeping 1.0s
    [api]   AMZN_2015-05-01.json  → sleeping 1.0s
    [api]   AMZN_2015-06-01.json  → sleeping 1.0s
    [api]   AMZN_2015-07-01.json  → sleeping 1.0s
    [api]   AMZN_2015-08-03.json  → sleeping 1.0s
    [api]   AMZN_2015-09-01.json  → sleeping 1.0s
    [api]   AMZN_2015-10-01.json  → sleeping 1.0s
    [api]   AMZN_2015-11-02.json  → sleeping 1.0s
    [api]   AMZN_2015-12-01.json  → sleeping 1.0s
    [api]   AMZN_2016-01-01.json  → sleeping 1.0s
    [api]   AMZN_2016-02-01.json  → sleeping 1.0s
    [api]   AMZN_2016-03-01.json  → sleeping 1.0s
    [api]   AMZN_2016-04-01.json  → sleeping 1.0s
    [api]   AMZN_2016-05-02.json  → sleeping 1.0s
    [api]   AMZN_2016-06-01.json  → sleeping 1.0s
    [api]   AMZN_2016-07-01.json  → sleeping 1.0s
    [api]   AMZN_2016-08-01.json  → sleeping 1.0s


  GOOG:   0%|          | 0/132 [00:00<?, ?it/s]

    [api]   GOOG_2015-01-01.json  → sleeping 1.0s
    [api]   GOOG_2015-02-02.json  → sleeping 1.0s
    [api]   GOOG_2015-03-02.json  → sleeping 1.0s
    [api]   GOOG_2015-04-01.json  → sleeping 1.0s
    [api]   GOOG_2015-05-01.json  → sleeping 1.0s
    [api]   GOOG_2015-06-01.json  → sleeping 1.0s
    [api]   GOOG_2015-07-01.json  → sleeping 1.0s
    [api]   GOOG_2015-08-03.json  → sleeping 1.0s
    [api]   GOOG_2015-09-01.json  → sleeping 1.0s
    [api]   GOOG_2015-10-01.json  → sleeping 1.0s
    [api]   GOOG_2015-11-02.json  → sleeping 1.0s
    [api]   GOOG_2015-12-01.json  → sleeping 1.0s
    [api]   GOOG_2016-01-01.json  → sleeping 1.0s
    [api]   GOOG_2016-02-01.json  → sleeping 1.0s
    [api]   GOOG_2016-03-01.json  → sleeping 1.0s
    [api]   GOOG_2016-04-01.json  → sleeping 1.0s
    [api]   GOOG_2016-05-02.json  → sleeping 1.0s
    [api]   GOOG_2016-06-01.json  → sleeping 1.0s
    [api]   GOOG_2016-07-01.json  → sleeping 1.0s
    [api]   GOOG_2016-08-01.json  → sleeping 1.0s


  META:   0%|          | 0/132 [00:00<?, ?it/s]

    [api]   META_2015-01-01.json  → sleeping 1.0s
    [api]   META_2015-02-02.json  → sleeping 1.0s
    [api]   META_2015-03-02.json  → sleeping 1.0s
    [api]   META_2015-04-01.json  → sleeping 1.0s
    [api]   META_2015-05-01.json  → sleeping 1.0s
    [api]   META_2015-06-01.json  → sleeping 1.0s
    [api]   META_2015-07-01.json  → sleeping 1.0s
    [api]   META_2015-08-03.json  → sleeping 1.0s
    [api]   META_2015-09-01.json  → sleeping 1.0s
    [api]   META_2015-10-01.json  → sleeping 1.0s
    [api]   META_2015-11-02.json  → sleeping 1.0s
    [api]   META_2015-12-01.json  → sleeping 1.0s
    [api]   META_2016-01-01.json  → sleeping 1.0s
    [api]   META_2016-02-01.json  → sleeping 1.0s
    [api]   META_2016-03-01.json  → sleeping 1.0s
    [api]   META_2016-04-01.json  → sleeping 1.0s
    [api]   META_2016-05-02.json  → sleeping 1.0s
    [api]   META_2016-06-01.json  → sleeping 1.0s
    [api]   META_2016-07-01.json  → sleeping 1.0s
    [api]   META_2016-08-01.json  → sleeping 1.0s


In [33]:
import json

opt_dfs = []

for sym in TICKERS:
    sym_dir = DIRS["raw_options"] / sym
    if not sym_dir.exists():
        print(f"  {sym}: directory not found")
        continue

    sym_dfs = []
    for cache in sorted(sym_dir.glob("*.json")):
        raw = json.loads(cache.read_text())
        if not raw.get("data"):
            continue

        df = pd.DataFrame(raw["data"])

        # Rename columns to standard names
        df = df.rename(columns={
            "date"              : "trade_date",
            "type"              : "call_put",
            "implied_volatility": "implied_vol",
        })

        # Parse dates
        for col in ["trade_date", "expiration"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        # Parse numerics
        num_cols = ["strike", "last", "mark", "bid", "ask",
                    "bid_size", "ask_size", "volume", "open_interest",
                    "implied_vol", "delta", "gamma", "theta", "vega", "rho"]
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        sym_dfs.append(df)

    if sym_dfs:
        sym_df = pd.concat(sym_dfs, ignore_index=True)
        sym_df.to_parquet(DIRS["proc_options"] / f"{sym}_options.parquet", index=False)
        sym_df.to_csv(DIRS["proc_options"]    / f"{sym}_options.csv",     index=False)
        opt_dfs.append(sym_df)
        print(f"  {sym}: {len(sym_df):,} rows across {len(sym_dfs)} months")
    else:
        print(f"  {sym}: no data in cache files")

if opt_dfs:
    ALL_options = pd.concat(opt_dfs, ignore_index=True)
    ALL_options.to_parquet(DIRS["proc_options"] / "ALL_options.parquet", index=False)
    ALL_options.to_csv(DIRS["proc_options"]    / "ALL_options.csv",     index=False)
    print(f"\nTotal options rows: {len(ALL_options):,}")

  ADMA: 7,602 rows across 79 months
  NTRA: 16,918 rows across 111 months
  AXON: 38,082 rows across 92 months
  SHAK: 90,620 rows across 111 months
  AAPL: 211,937 rows across 118 months
  MSFT: 210,564 rows across 118 months
  NVDA: 319,078 rows across 118 months
  AMZN: 459,512 rows across 118 months
  GOOG: 339,085 rows across 118 months
  META: 160,624 rows across 43 months

Total options rows: 1,854,022


In [34]:
#  Combine & Save All Options 
if opt_dfs:
    options_all = pd.concat(opt_dfs, ignore_index=True)
    options_all.to_parquet(DIRS["proc_options"] / "ALL_options.parquet", index=False)
    options_all.to_csv(DIRS["proc_options"]    / "ALL_options.csv",     index=False)

    print(f" Combined options saved: {options_all.shape}")
    print(f"\nColumns: {list(options_all.columns)}")
    print(f"\nDate coverage per ticker:")
    print(
        options_all.groupby("symbol")["trade_date"]
        .agg(first="min", last="max", months="nunique")
    )
    print(f"\nCall/Put split:")
    if "call_put" in options_all.columns:
        print(options_all["call_put"].str.upper().value_counts())
else:
    options_all = pd.DataFrame()
    print(" No options data collected.")

 Combined options saved: (1854022, 20)

Columns: ['contractID', 'symbol', 'expiration', 'strike', 'call_put', 'last', 'mark', 'bid', 'bid_size', 'ask', 'ask_size', 'volume', 'open_interest', 'trade_date', 'implied_vol', 'delta', 'gamma', 'theta', 'vega', 'rho']

Date coverage per ticker:
            first       last  months
symbol                              
AAPL   2015-02-02 2025-12-01     118
ADMA   2018-08-01 2025-12-01      79
AMZN   2015-02-02 2025-12-01     118
AXON   2015-07-01 2025-12-01      92
GOOG   2015-02-02 2025-12-01     118
META   2021-08-02 2025-12-01      43
MSFT   2015-02-02 2025-12-01     118
NTRA   2015-09-01 2025-12-01     111
NVDA   2015-02-02 2025-12-01     118
SHAK   2015-09-01 2025-12-01     111

Call/Put split:
call_put
PUT     927014
CALL    927008
Name: count, dtype: int64


## Pull Summary

In [35]:
print("=" * 60)
print("DATA PULL SUMMARY")
print("=" * 60)
print(f"Universe       : {TICKERS}")
print(f"Date Range     : {DATE_START} → {DATE_END}")
print()
print(f"Daily rows     : {len(combined_daily):,}")
print(f"Income qtrs    : {len(income_all):,}")
print(f"Balance qtrs   : {len(balance_all):,}")
print(f"Cashflow qtrs  : {len(cashflow_all):,}")
print(f"Overview rows  : {len(overview_all)}")
if opt_dfs:
    print(f"Options rows   : {len(options_all):,}  ({options_all['trade_date'].nunique()} dates X {options_all['symbol'].nunique()} tickers)")
else:
    print(f"Options rows   : 0")
print()
print("All raw JSON cached in  : data/raw/")
print("Parsed files saved in   : data/processed/")
print("=" * 60)


DATA PULL SUMMARY
Universe       : ['ADMA', 'NTRA', 'AXON', 'SHAK', 'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOG', 'META']
Date Range     : 2015-01-01 → 2025-12-31

Daily rows     : 27,516
Income qtrs    : 736
Balance qtrs   : 729
Cashflow qtrs  : 731
Overview rows  : 10
Options rows   : 1,854,022  (118 dates X 10 tickers)

All raw JSON cached in  : data/raw/
Parsed files saved in   : data/processed/
